<a href="https://colab.research.google.com/github/peacemac/codeatom/blob/main/logReco.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
# LOG Recovery Engine Test code #
import torch
import torch.nn as nn
from collections import Counter
import random

# ==========================================
# 1. Prepare the Infrastructure Log Corpus
# ==========================================
# A sample of typical server and infrastructure logs
LOG_CORPUS = [
    "service docker failed to start",
    "vmware esxi host is unresponsive",
    "executing powershell backup script successfully",
    "service apache2 restarted successfully",
    "failed to connect to vmware vcenter",
    "docker container exited with error code",
    "powershell script deployment completed",
    "critical error in vmware esxi storage"
]

SPECIAL_TOKENS = ["[PAD]", "[MASK]", "[UNK]"]

# Build frequency-sorted vocabulary (Required for Adaptive Softmax)
words = " ".join(LOG_CORPUS).split()
word_counts = Counter(words)
sorted_words = [word for word, count in word_counts.most_common()]
vocab = SPECIAL_TOKENS + sorted_words

word2idx = {w: i for i, w in enumerate(vocab)}
idx2word = {i: w for i, w in enumerate(vocab)}
VOCAB_SIZE = len(vocab)

# ==========================================
# 2. Define the Optimized MLM Architecture
# ==========================================
class LogRecoveryMLM(nn.Module):
    def __init__(self, vocab_size, d_model=64, nhead=2, num_layers=2):
        super(LogRecoveryMLM, self).__init__()
        self.d_model = d_model

        self.embedding = nn.Embedding(vocab_size, d_model, padding_idx=word2idx["[PAD]"])
        encoder_layer = nn.TransformerEncoderLayer(d_model=d_model, nhead=nhead, batch_first=True)
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)

        # Adaptive Softmax for efficiency
        # (Cutoffs scaled way down for this tiny toy dictionary)
        self.adaptive_softmax = nn.AdaptiveLogSoftmaxWithLoss(
            in_features=d_model,
            n_classes=vocab_size,
            cutoffs=[10, 20, 30], # In production, these would be [5000, 20000, 50000]
            div_value=2.0
        )

    def forward(self, x, targets=None):
        embedded = self.embedding(x)
        hidden = self.transformer(embedded)
        flat_hidden = hidden.view(-1, self.d_model)

        if targets is not None:
            flat_targets = targets.view(-1)
            out = self.adaptive_softmax(flat_hidden, flat_targets)
            return out.loss
        else:
            predictions = self.adaptive_softmax.predict(flat_hidden)
            return predictions

# ==========================================
# 3. Training the Model
# ==========================================
def encode(sentence):
    return [word2idx.get(w, word2idx["[UNK]"]) for w in sentence.split()]

model = LogRecoveryMLM(VOCAB_SIZE)
optimizer = torch.optim.Adam(model.parameters(), lr=0.005)

print("Training Log Recovery Model...")
epochs = 60
for epoch in range(epochs):
    inputs, targets = [], []

    # Mask random words to teach the model context
    for log in LOG_CORPUS:
        encoded = encode(log)
        inp, tgt = encoded.copy(), encoded.copy()

        # Pick one random word to mask per log
        mask_idx = random.randint(0, len(encoded) - 1)
        inp[mask_idx] = word2idx["[MASK]"]

        inputs.append(inp)
        targets.append(tgt)

    # Pad sequences so they fit in a single batch matrix
    max_len = max(len(seq) for seq in inputs)
    inputs_padded = [seq + [word2idx["[PAD]"]] * (max_len - len(seq)) for seq in inputs]
    targets_padded = [seq + [word2idx["[PAD]"]] * (max_len - len(seq)) for seq in targets]

    x_tensor = torch.tensor(inputs_padded)
    y_tensor = torch.tensor(targets_padded)

    optimizer.zero_grad()
    loss = model(x_tensor, targets=y_tensor)
    loss.backward()
    optimizer.step()

    if (epoch + 1) % 15 == 0:
        print(f"Epoch {epoch+1}/{epochs} | Loss: {loss.item():.4f}")

# ==========================================
# 4. Inference: Recovering Corrupted Logs
# ==========================================
def recover_log(corrupted_log):
    model.eval()
    tokens = corrupted_log.split()
    if "[MASK]" not in tokens:
        return "No [MASK] found in log."

    mask_pos = tokens.index("[MASK]")
    encoded = torch.tensor([encode(corrupted_log)])

    with torch.no_grad():
        predicted_indices = model(encoded)

    # Reshape predictions back to (batch_size, seq_length) and grab the masked token
    predicted_indices = predicted_indices.view(1, -1)
    predicted_word_idx = predicted_indices[0, mask_pos].item()

    # Reconstruct the log
    tokens[mask_pos] = f">>>{idx2word[predicted_word_idx].upper()}<<<"
    return " ".join(tokens)

print("\n--- Testing Log Recovery ---")
test_logs = [
    "service [MASK] failed to start",
    "failed to connect to vmware [MASK]",
    "executing [MASK] backup script successfully"
]

for log in test_logs:
    print(f"Corrupted: {log}")
    print(f"Recovered: {recover_log(log)}\n")

Training Log Recovery Model...
Epoch 15/60 | Loss: 1.0121
Epoch 30/60 | Loss: 0.3852
Epoch 45/60 | Loss: 0.4800
Epoch 60/60 | Loss: 0.2978

--- Testing Log Recovery ---
Corrupted: service [MASK] failed to start
Recovered: service >>>TO<<< failed to start

Corrupted: failed to connect to vmware [MASK]
Recovered: failed to connect to vmware >>>TO<<<

Corrupted: executing [MASK] backup script successfully
Recovered: executing >>>POWERSHELL<<< backup script successfully

